# store

> Arrays built once and shared: memory-mapped from disk by every process that needs them, and an inverted index over keys FTS5 cannot express.

In [ ]:
#| default_exp store

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
import tempfile, subprocess, sys

In [ ]:
#| export
import fcntl, os, shutil
import numpy as np
from array import array
from bisect import bisect_left
from fastcore.all import Path

## Stored arrays

`stored` returns the arrays `build()` makes, as a dict of name to array, memory-mapped from `root/<name>-<key>/`. The first process to need them builds them under a lock and writes them to a temporary folder it then renames into place, so a reader never maps a half-written set and processes starting together build once. The mapped files are pages the processes share. `key` names what the arrays are built from, such as an index's `fts_sig` and the library versions; a new key builds afresh and removes the older sets (a process still mapping one keeps it, since an unlinked file stays mapped). When `root` cannot be written, the arrays are built in memory.

In [ ]:
#| export
def stored(root,   # folder holding every stored set
           name,   # what the arrays are, the prefix of their folder
           key,    # what they are built from; a new key builds again
           build   # `() -> {name: array}`
          ) -> dict:
    "`build()`'s arrays, memory-mapped from `root/<name>-<key>`, built once across processes."
    root = Path(root); d = root/f'{name}-{key}'
    try:
        if not d.is_dir():
            root.mkdir(parents=True, exist_ok=True)
            with open(root/f'.{name}.lock', 'w') as lk:
                fcntl.flock(lk, fcntl.LOCK_EX)
                if not d.is_dir():
                    tmp = root/f'.{d.name}.{os.getpid()}'
                    tmp.mkdir(exist_ok=True)
                    for k, v in build().items(): np.save(tmp/f'{k}.npy', np.asarray(v))
                    os.rename(tmp, d)
                    for o in root.glob(f'{name}-*'):
                        if o != d: shutil.rmtree(o, ignore_errors=True)
        return {p.stem: np.load(p, mmap_mode='r') for p in d.glob('*.npy')}
    except OSError: return {k: np.asarray(v) for k, v in build().items()}

In [ ]:
_root = Path(tempfile.mkdtemp())
_n = [0]
def _b(): _n[0] += 1; return dict(a=np.arange(5), b=np.array(['x', 'y']))
_s = stored(_root, 'demo', 'k1', _b)
test_eq(_s['a'].tolist(), [0, 1, 2, 3, 4]); assert isinstance(_s['a'], np.memmap)
stored(_root, 'demo', 'k1', _b); test_eq(_n[0], 1)            # built once
stored(_root, 'demo', 'k2', _b); test_eq(_n[0], 2)
test_eq(sorted(p.name for p in _root.glob('demo-*')), ['demo-k2'])   # the older set is gone

In [ ]:
_pr = Path(tempfile.mkdtemp())
_src = f"""import numpy as np, time; from pathlib import Path; from litesearch.store import stored
def b(): open(Path({str(_pr)!r})/'built', 'a').write('x'); time.sleep(.3); return dict(a=np.ones(3))
stored({str(_pr)!r}, 'par', 'k', b)"""
_ps = [subprocess.Popen([sys.executable, '-c', _src]) for _ in range(3)]
test_eq([p.wait() for p in _ps], [0, 0, 0])
test_eq((_pr/'built').read_text(), 'x')                      # three processes, one build

## A key index

`KeyIndex` is an inverted index from string keys to row numbers, for keys FTS5's tokens cannot express: a folded spelling, an alias, a romanisation. `KeyIndex.build(pairs)` takes `(key, row)` pairs and returns its arrays, which `stored` can keep; `KeyIndex(arrays)` reads them. The keys are sorted and packed into one UTF-8 blob, so `lookup` is a bisect: the rows holding `key`, or with `prefix=True` every key that `key` begins, in row order without repeats.

In [ ]:
#| export
class KeyIndex:
    "Rows by string key, exact or by prefix, over arrays `build` makes."
    def __init__(self, a): self.w, self.ws, self.s, self.p = a['w'], a['ws'], a['s'], a['p']
    def __len__(self): return len(self.ws) - 1
    def __getitem__(self, i): return self.w[self.ws[i]:self.ws[i + 1]].tobytes()
    def keys(self): return [self[i].decode() for i in range(len(self))]
    @staticmethod
    def build(pairs) -> dict:
        "Arrays for `(key, row)` pairs."
        ks, rs = zip(*pairs) if (pairs := list(pairs)) else ((), ())
        u = sorted(set(ks)); at = {k: i for i, k in enumerate(u)}
        ids = np.fromiter((at[k] for k in ks), np.int64, len(ks)); o = np.argsort(ids, kind='stable')
        b, st = bytearray(), array('q')
        for k in u: st.append(len(b)); b += k.encode()
        st.append(len(b))
        return dict(w=np.frombuffer(bytes(b), np.uint8), ws=np.frombuffer(st, np.int64),
                    s=np.searchsorted(ids[o], np.arange(len(u) + 1)), p=np.asarray(rs, np.int64)[o])
    def lookup(self, key, prefix=False) -> np.ndarray:
        "Rows holding `key`, or with `prefix` any key `key` begins."
        b = key.encode(); lo = bisect_left(self, b)
        hi = bisect_left(self, b + b'\xff', lo) if prefix else lo + (lo < len(self) and self[lo] == b)
        return np.unique(self.p[self.s[lo]:self.s[hi]])

In [ ]:
_k = KeyIndex(KeyIndex.build([('krsna', 0), ('krsna', 2), ('rama', 1), ('ramayana', 3), ('कृष्ण', 2)]))
test_eq(_k.lookup('krsna').tolist(), [0, 2])
test_eq(_k.lookup('rama').tolist(), [1])
test_eq(_k.lookup('rama', prefix=True).tolist(), [1, 3])
test_eq(_k.lookup('कृ', prefix=True).tolist(), [2])
test_eq(_k.lookup('sita').tolist(), [])
test_eq(len(KeyIndex(KeyIndex.build([]))), 0)
_ks = KeyIndex(stored(_root, 'keys', 'k', lambda: KeyIndex.build([('a', 1), ('ab', 2)])))
test_eq(_ks.lookup('a', prefix=True).tolist(), [1, 2])

## A vector matrix

`VecMatrix` holds a store's embeddings as one unit-normalised float32 matrix and searches it by a matrix product: exact cosine, with no index to build or keep in step. `VecMatrix.build(tbl)` reads the vectors and returns its arrays; kept by `stored`, every process maps one copy. It suits a small or medium store; past a few hundred thousand rows an ANN index (`get_index`) is faster. Embeddings carry no FTS signature, so the caller names the key: a count and the highest rowid, or a token its own writes move.

In [ ]:
#| export
class VecMatrix:
    "Exact cosine search over a store's embeddings, as one unit float32 matrix."
    def __init__(self, a): self.ids, self.m = a['ids'], a['m']
    def __len__(self): return len(self.ids)
    @staticmethod
    def build(tbl, emb_col='embedding', dtype=np.float16, where=None) -> dict:
        "Arrays for `tbl`'s rows with an embedding, optionally narrowed by `where`."
        rs = tbl.db.q(f'select rowid, [{emb_col}] e from [{tbl.name}] where [{emb_col}] is not null' + (f' and {where}' if where else ''))
        if not rs: return dict(ids=np.zeros(0, np.int64), m=np.zeros((0, 0), np.float32))
        m = np.stack([np.frombuffer(r['e'], dtype) for r in rs]).astype(np.float32)
        m /= np.maximum(np.linalg.norm(m, axis=1, keepdims=True), 1e-12)
        return dict(ids=np.array([r['rowid'] for r in rs], np.int64), m=m)
    def search(self, q, k=10, dtype=np.float16) -> list:
        "`(rowid, similarity)` for the `k` rows nearest `q`, bytes or an array."
        if not len(self): return []
        q = (np.frombuffer(q, dtype) if isinstance(q, (bytes, bytearray, memoryview)) else np.asarray(q)).astype(np.float32)
        s = self.m @ (q / max(np.linalg.norm(q), 1e-12))
        top = np.argpartition(-s, min(k, len(s)) - 1)[:k]
        return [(int(self.ids[i]), float(s[i])) for i in top[np.argsort(-s[top])]]

In [ ]:
from litesearch.core import database
_vd = database(sem_search=False)
_vd.execute('create table v(name text, embedding blob)')
for nm, e in [('a', [1, 0, 0]), ('b', [0, 1, 0]), ('c', [1, 1, 0]), ('none', None)]:
    _vd.execute('insert into v values(?, ?)', [nm, None if e is None else np.array(e, np.float16).tobytes()])
_vm = VecMatrix(VecMatrix.build(_vd.t.v))
test_eq(len(_vm), 3)
_hits = _vm.search(np.array([1, .1, 0], np.float16).tobytes(), k=2)
test_eq([i for i, _ in _hits], [1, 3]); assert _hits[0][1] > _hits[1][1]
test_eq(VecMatrix(VecMatrix.build(_vd.t.v, where="name='b'")).search([0, 1, 0], k=5)[0][0], 2)
_vs = VecMatrix(stored(_root, 'vecs', 'k', lambda: VecMatrix.build(_vd.t.v)))
assert isinstance(_vs.m, np.memmap) and _vs.search([0, 1, 0], k=1)[0][0] == 2
test_eq(VecMatrix(VecMatrix.build(_vd.t.v, where='0')).search([1, 0, 0]), [])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()